In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.text("catalog", "workspace", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "live_transit_monitor", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("bronze_schema")
SILVER = f"{CATALOG}.{SCHEMA}.gps_positions_silver"
SILVER_CHECKED = f"{CATALOG}.{SCHEMA}.gps_positions_silver_checked"
SILVER_VALID = f"{CATALOG}.{SCHEMA}.gps_positions_silver_valid"
BRONZE = f"{CATALOG}.{SCHEMA}.gps_data"

SILVER_QUARANTINE = f"{CATALOG}.{SCHEMA}.gps_positions_quarantine"
#small quality metrics table
GOLD_DQ_TREND = f"{CATALOG}.{SCHEMA}.gold_dq_trend"
GOLD_DQ_REASONS = f"{CATALOG}.{SCHEMA}.gold_dq_reasons"
GOLD_DQ_BY_REASON = f"{CATALOG}.{SCHEMA}.gold_dq_by_reason"
GOLD_DQ_WARNINGS = f"{CATALOG}.{SCHEMA}.gold_dq_warnings"
GOLD_DQ_TOP_OFFENDERS = f"{CATALOG}.{SCHEMA}.gold_dq_top_offenders"
GOLD_RECONCILIATION = f"{CATALOG}.{SCHEMA}.gold_reconciliation"

In [0]:
silver_checked = spark.read.table(SILVER_CHECKED)
silver = spark.read.table(SILVER)
silver_valid = spark.read.table(SILVER_VALID)
silver_quarantine = spark.read.table(SILVER_QUARANTINE)


In [0]:
gold_dq_trend = (
    silver_checked
    .groupBy(F.date_trunc("hour", "event_time_local").alias("hour_ts"))
    .agg(
        F.count("*").alias("total_records"),
        F.count(F.when(F.col("_errors").isNotNull(), True)).alias("quarantined_records"),
        F.count(F.when(F.col("_errors").isNull(), True)).alias("valid_records"),
        F.count(F.when(F.col("_warnings").isNotNull() & F.col("_errors").isNull(), True)).alias("warn_records"),
        F.countDistinct("vehicleId").alias("distinct_vehicles"),
        F.countDistinct(F.when(F.col("_errors").isNotNull(), F.col("vehicleId"))).alias("distinct_vehicles_with_errors"),
    )
    .withColumn("quarantine_rate_pct", F.round(F.col("quarantined_records") / F.col("total_records") * 100, 2))
    .withColumn("warn_rate_pct",       F.round(F.col("warn_records")        / F.col("total_records") * 100, 2))
)
#display(gold_dq_trend)
(gold_dq_trend.write
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(GOLD_DQ_TREND))

In [0]:
gold_dq_reasons = (
    silver_quarantine
    .withColumn("error_struct", F.explode("_errors"))
    .select(
        F.col("error_struct.name").alias("check_name"),
        F.col("error_struct.message").alias("error_message"),
        F.col("vehicleId"),
        F.col("tripId"),
        F.col("event_time_local")
    )
)

#display(gold_dq_reasons)
(gold_dq_reasons.write
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(GOLD_DQ_REASONS))

In [0]:
gold_dq_by_reason = (
    gold_dq_reasons
    .groupBy("check_name")
    .agg(F.count("*").alias("failed_records"))
    .orderBy(F.desc("failed_records"))
)
#display(gold_dq_by_reason)
(gold_dq_by_reason.write
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(GOLD_DQ_BY_REASON))

In [0]:
gold_dq_warnings = (
    silver_checked                                 
    .withColumn("w", F.explode("_warnings"))
    .groupBy(F.col("w.name").alias("check_name"))
    .agg(F.count("*").alias("warned_records"))
    .orderBy(F.desc("warned_records"))
)

#display(gold_dq_warnings)
(gold_dq_warnings.write
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(GOLD_DQ_WARNINGS))

In [0]:
gold_top_offenders = (
    silver_checked
    .groupBy("vehicleCode", "vehicleId")
    .agg(
        F.count("*").alias("total_readings"),
        F.count(F.when(F.col("_errors").isNotNull(), True)).alias("error_readings"),
        F.count(F.when(F.col("_warnings").isNotNull(), True)).alias("warn_readings"),
    )
    # bad = all problems flagged by DQX; warn = poor GPS signal quality (gpsQuality < 1)
    .withColumn("bad_readings", F.col("error_readings") + F.col("warn_readings"))
    .withColumn("warn_rate_pct",  F.round(F.col("warn_readings")  / F.col("total_readings") * 100, 2))
    .withColumn("error_rate_pct", F.round(F.col("error_readings") / F.col("total_readings") * 100, 2))
    .withColumn("bad_rate_pct",   F.round(F.col("bad_readings")   / F.col("total_readings") * 100, 2))
    # only vehicles with at least 5 readings and at least one problem
    .filter((F.col("total_readings") >= 5) & (F.col("bad_readings") > 0))
    .orderBy(F.desc("bad_rate_pct"), F.desc("bad_readings"))
    .limit(10)
)
#display(gold_top_offenders)
(gold_top_offenders.write
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(GOLD_DQ_TOP_OFFENDERS))

         

In [0]:
bronze = spark.read.table(BRONZE)

In [0]:
bronze_count     = bronze.count()
checked_count    = silver_checked.count()
quarantine_count = silver_quarantine.count()
valid_count      = silver_valid.count()
silver_count     = silver.count()

gold_reconciliation = (
    spark.createDataFrame(
        [(bronze_count, checked_count, quarantine_count, valid_count, silver_count)],
        ["bronze_count", "checked_count", "quarantine_count", "valid_count", "silver_count"],
    )
    .withColumn("split_ok",          (F.col("valid_count") + F.col("quarantine_count")) == F.col("checked_count"))
    .withColumn("checked_eq_bronze", F.col("checked_count") == F.col("bronze_count"))
    .withColumn("dedup_removed",     F.col("valid_count") - F.col("silver_count"))
    .withColumn("quarantine_rate_pct", F.round(F.col("quarantine_count") / F.col("checked_count") * 100, 2))
    .withColumn("snapshot_time",     F.current_timestamp())
)

#display(gold_reconciliation)
(gold_reconciliation.write
    .mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(GOLD_RECONCILIATION))